In [19]:
#  imports
import pandas as pd
import numpy as np

import re
import string

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

In [20]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [21]:
from google.colab import files

uploaded = files.upload()

Saving train_data.txt to train_data (1).txt


In [22]:
train_df = pd.read_csv(
    "train_data.txt",
    sep=" ::: ",
    engine="python",
    header=None,
    names=["ID", "Title", "Genre", "Description"]
)

In [23]:
train_df.head()

,ID,Title,Genre,Description
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


In [24]:
# Create stopword list and lemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [25]:
# Function to convert text to lowercase
def to_lowercase(text):
    return text.lower()


# Function to remove HTML tags
def remove_html(text):
    return re.sub(r"<.*?>", "", text)


# Function to remove URLs
def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", "", text)


# Function to remove numbers
def remove_numbers(text):
    return re.sub(r"\d+", "", text)


# Function to remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))


# Function to remove extra spaces
def remove_extra_spaces(text):
    return re.sub(r"\s+", " ", text).strip()


# Function to remove stopwords
def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)


# Function to perform lemmatization
def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

In [26]:
def preprocess_text(text):
    """
    Perform complete text preprocessing.
    """

    text = to_lowercase(text)
    text = remove_html(text)
    text = remove_urls(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = remove_extra_spaces(text)
    text = remove_stopwords(text)
    text = lemmatize_text(text)

    return text

In [27]:
sample = train_df.loc[0, "Description"]

print("Original Text:\n")
print(sample)

print("\n" + "="*100 + "\n")

print("Clean Text:\n")
print(preprocess_text(sample))

Original Text:

Listening in to a conversation between his doctor and parents, 10-year-old Oscar learns what nobody has the courage to tell him. He only has a few weeks to live. Furious, he refuses to speak to anyone except straight-talking Rose, the lady in pink he meets on the hospital stairs. As Christmas approaches, Rose uses her fantastical experiences as a professional wrestler, her imagination, wit and charm to allow Oscar to live life and love to the full, in the company of his friends Pop Corn, Einstein, Bacon and childhood sweetheart Peggy Blue.


Clean Text:

listening conversation doctor parent yearold oscar learns nobody courage tell week live furious refuse speak anyone except straighttalking rose lady pink meet hospital stair christmas approach rose us fantastical experience professional wrestler imagination wit charm allow oscar live life love full company friend pop corn einstein bacon childhood sweetheart peggy blue


In [28]:
# Apply preprocessing to all movie descriptions

train_df["clean_description"] = train_df["Description"].apply(preprocess_text)

In [29]:
train_df[["Description", "clean_description"]].head()

,Description,clean_description
0,Listening in to a conversation between his doc...,listening conversation doctor parent yearold o...
1,A brother and sister with a past incestuous re...,brother sister past incestuous relationship cu...
2,As the bus empties the students for their fiel...,bus empty student field trip museum natural hi...
3,To help their unemployed father make ends meet...,help unemployed father make end meet edith twi...
4,The film's title refers not only to the un-rec...,film title refers unrecovered body ground zero...


In [30]:
train_df["clean_description"].isnull().sum()

np.int64(0)

In [31]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2
)

In [32]:
X = tfidf.fit_transform(train_df["clean_description"])

y = train_df["Genre"]

In [33]:
print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (54214, 5000)
Target Shape: (54214,)


In [35]:
# # Notebook Summary

# ### Tasks Completed

# - Loaded the training dataset
# - Performed text preprocessing
# - Converted text to lowercase
# - Removed HTML tags, URLs, numbers, and punctuation
# - Removed stopwords
# - Applied lemmatization
# - Generated a cleaned text column
# - Converted text into TF-IDF feature vectors

# The dataset is now ready for machine learning model training.